# Qubit-to-QuQuart Circuit Mapper: Examples
Each cell below is a self-contained example. Run the **Imports** cell first, then run any example cell independently.

In [1]:
# Imports — run this cell first
import numpy as np
import qutip as qt
from qubit_to_ququart_mapper import (
    QubitCircuit,
    best_mapping_optimization,
    qubit_to_ququart_circuit,
    create_example_circuit,
    display_ququart_circuit,
    KNOWN_GATES,
)

## Example 1: Basic Usage with Automatic Mapping Optimization
4-qubit circuit with strong intra-pair coupling. BMO should find `[(0,1), (2,3)]`.

In [2]:
circuit = QubitCircuit(num_qubits=4)

H    = KNOWN_GATES["H"]
CNOT = KNOWN_GATES["CNOT"]

# Strong coupling between (q0,q1) and (q2,q3)
circuit.add_layer([(H, [i]) for i in range(4)])
circuit.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3])])
circuit.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3])])
circuit.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3])])

ququart_circuit = qubit_to_ququart_circuit(circuit)
display_ququart_circuit(ququart_circuit)

print(f"\nResult: {ququart_circuit}")
print(f"Expected mapping: [(0,1), (2,3)] due to high interaction within pairs")


No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {(0, 1): 3, (2, 3): 3}

  Most optimal pairing: [(0, 1), (2, 3)]
  Cross-qu-quart gates: 0

Converting 4 layers...

Conversion complete!
  Total layers: 4
  Cross-qu-quart gates: 0

QuQuart Circuit Details

Qubit-to-QuQuart Mapping (Optimal):

  QQ0: qubit 0 (pos 0) + qubit 1 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ0 = |0>_q0 |0>_q1  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ0 = |1>_q0 |0>_q1  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ0 = |0>_q0 |1>_q1  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ0 = |1>_q0 |1>_q1  (pos0=1, pos1=1 -> 1+2=3)

  QQ1: qubit 2 (pos 0) + qubit 3 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ1 = |0>_q2 |0>_q3  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ1 = |1>_q2 |0>_q3  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ1 = |0>_q2 |1>_q3  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ1 = |1>_q2 |1>_q3  (pos0=1, pos1=1 -> 1+2=3)

  Total ququarts: 2
  Total layers:   4
  Cross-ququar

## Example 2: Manual Mapping Specification
Same circuit as Example 1 but with a user-specified (suboptimal) mapping.

In [3]:
circuit = create_example_circuit()  # 4-qubit example circuit

custom_mapping = [(0, 2), (1, 3)]   # pair q0 with q2, q1 with q3

print(f"Using custom mapping: {custom_mapping}")
ququart_circuit = qubit_to_ququart_circuit(circuit, mapping=custom_mapping)

print(f"\nResult: {ququart_circuit}")
print(f"Note: this mapping may result in more cross-qu-quart gates")

Using custom mapping: [(0, 2), (1, 3)]
Using provided mapping: [(0, 2), (1, 3)]

Converting 3 layers...
  Layer 1: Cross-qu-quart gate on qubits [0, 3] -> qu-quarts [0, 1]
  Layer 1: Cross-qu-quart gate on qubits [1, 2] -> qu-quarts [0, 1]
  Layer 2: Cross-qu-quart gate on qubits [2, 3] -> qu-quarts [0, 1]

Conversion complete!
  Total layers: 3
  Cross-qu-quart gates: 3

Result: QuQuartCircuit(2 qu-quarts from 4 qubits, 3 layers, 3 cross-qu-quart gates)
Note: this mapping may result in more cross-qu-quart gates


## Example 3: Different Interaction Topologies
Compares linear chain, star, and complete-graph connectivity on 4 qubits.

In [4]:
CNOT = KNOWN_GATES["CNOT"]

# Topology A: Linear chain (q0-q1-q2-q3)
print("Topology A: Linear Chain (q0-q1-q2-q3)")
circuit_a = QubitCircuit(num_qubits=4)
circuit_a.add_layer([(CNOT, [0, 1])])
circuit_a.add_layer([(CNOT, [1, 2])])
circuit_a.add_layer([(CNOT, [2, 3])])
ququart_a = qubit_to_ququart_circuit(circuit_a)
print(f"Result: {ququart_a}")

# Topology B: Star (all connect to q0)
print("\nTopology B: Star (all qubits connect to q0)")
circuit_b = QubitCircuit(num_qubits=4)
circuit_b.add_layer([(CNOT, [0, 1])])
circuit_b.add_layer([(CNOT, [0, 2])])
circuit_b.add_layer([(CNOT, [0, 3])])
ququart_b = qubit_to_ququart_circuit(circuit_b)
print(f"Result: {ququart_b}")

# Topology C: Complete graph (all pairs interact equally)
print("\nTopology C: Complete Graph (all pairs interact)")
circuit_c = QubitCircuit(num_qubits=4)
circuit_c.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3])])
circuit_c.add_layer([(CNOT, [0, 2]), (CNOT, [1, 3])])
circuit_c.add_layer([(CNOT, [0, 3]), (CNOT, [1, 2])])
ququart_c = qubit_to_ququart_circuit(circuit_c)
print(f"Result: {ququart_c}")

Topology A: Linear Chain (q0-q1-q2-q3)

No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {(0, 1): 1, (1, 2): 1, (2, 3): 1}

  Most optimal pairing: [(0, 1), (2, 3)]
  Cross-qu-quart gates: 1

Converting 3 layers...
  Layer 1: Cross-qu-quart gate on qubits [1, 2] -> qu-quarts [0, 1]

Conversion complete!
  Total layers: 3
  Cross-qu-quart gates: 1
Result: QuQuartCircuit(2 qu-quarts from 4 qubits, 3 layers, 1 cross-qu-quart gates)

Topology B: Star (all qubits connect to q0)

No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {(0, 1): 1, (0, 2): 1, (0, 3): 1}

  Most optimal pairing: [(0, 1), (2, 3)]
  Cross-qu-quart gates: 2

Converting 3 layers...
  Layer 1: Cross-qu-quart gate on qubits [0, 2] -> qu-quarts [0, 1]
  Layer 2: Cross-qu-quart gate on qubits [0, 3] -> qu-quarts [0, 1]

Conversion complete!
  Total layers: 3
  Cross-qu-quart gates: 2
Result: QuQuartCircuit(2

## Example 4: Analyzing Optimization Benefit
Compares all three possible 2-ququart mappings to show BMO picks the best one.

In [5]:
CNOT = KNOWN_GATES["CNOT"]

circuit = QubitCircuit(num_qubits=4)

# Strong interaction between (0,1) and (2,3)
for _ in range(5):
    circuit.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3])])

# One cross-pair interaction
circuit.add_layer([(CNOT, [0, 2])])

print("Circuit has:")
print("  - 10 gates within pairs (0,1) and (2,3)")
print("  - 1 gate crossing pairs")

mappings = [
    [(0, 1), (2, 3)],  # optimal
    [(0, 2), (1, 3)],  # suboptimal
    [(0, 3), (1, 2)],  # suboptimal
]

results = []
for mapping in mappings:
    print(f"\n--- Testing mapping {mapping} ---")
    ququart = qubit_to_ququart_circuit(circuit, mapping=mapping)
    results.append((mapping, ququart.cross_ququart_gate_count))

print("\n" + "-"*50)
print("Comparison of Mappings:")
for mapping, cross_gates in results:
    print(f"  {mapping}: {cross_gates} cross-qu-quart gates")

best = min(results, key=lambda x: x[1])
print(f"\nBest mapping: {best[0]} with {best[1]} cross-qu-quart gates")

Circuit has:
  - 10 gates within pairs (0,1) and (2,3)
  - 1 gate crossing pairs

--- Testing mapping [(0, 1), (2, 3)] ---
Using provided mapping: [(0, 1), (2, 3)]

Converting 6 layers...
  Layer 5: Cross-qu-quart gate on qubits [0, 2] -> qu-quarts [0, 1]

Conversion complete!
  Total layers: 6
  Cross-qu-quart gates: 1

--- Testing mapping [(0, 2), (1, 3)] ---
Using provided mapping: [(0, 2), (1, 3)]

Converting 6 layers...
  Layer 0: Cross-qu-quart gate on qubits [0, 1] -> qu-quarts [0, 1]
  Layer 0: Cross-qu-quart gate on qubits [2, 3] -> qu-quarts [0, 1]
  Layer 1: Cross-qu-quart gate on qubits [0, 1] -> qu-quarts [0, 1]
  Layer 1: Cross-qu-quart gate on qubits [2, 3] -> qu-quarts [0, 1]
  Layer 2: Cross-qu-quart gate on qubits [0, 1] -> qu-quarts [0, 1]
  Layer 2: Cross-qu-quart gate on qubits [2, 3] -> qu-quarts [0, 1]
  Layer 3: Cross-qu-quart gate on qubits [0, 1] -> qu-quarts [0, 1]
  Layer 3: Cross-qu-quart gate on qubits [2, 3] -> qu-quarts [0, 1]
  Layer 4: Cross-qu-quart g

## Example 5: Edge Cases
**Case A** — circuit with only single-qubit gates (any mapping is equivalent).  
**Case B** — circuit where one qubit (q3) never participates in a 2-qubit gate.

In [6]:
H    = KNOWN_GATES["H"]
X    = KNOWN_GATES["X"]
CNOT = KNOWN_GATES["CNOT"]

# Case A: only single-qubit gates
print("Case A: Circuit with only single-qubit gates")
circuit_a = QubitCircuit(num_qubits=4)
circuit_a.add_layer([(H, [i]) for i in range(4)])
circuit_a.add_layer([(X, [i]) for i in range(4)])
ququart_a = qubit_to_ququart_circuit(circuit_a)
print(f"Result: {ququart_a}")
print("Note: with no 2-qubit gates, any mapping is equivalent")

# Case B: isolated qubit
print("\nCase B: Circuit with isolated qubit (q3 never used in 2-qubit gate)")
circuit_b = QubitCircuit(num_qubits=4)
circuit_b.add_layer([(CNOT, [0, 1])])
circuit_b.add_layer([(CNOT, [1, 2])])
circuit_b.add_layer([(H, [3])])
ququart_b = qubit_to_ququart_circuit(circuit_b)
print(f"Result: {ququart_b}")
print("Note: q3 will be paired with another qubit despite no 2-qubit interactions")

Case A: Circuit with only single-qubit gates

No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {}

  Most optimal pairing: [(0, 1), (2, 3)]
  Cross-qu-quart gates: 0

Converting 2 layers...

Conversion complete!
  Total layers: 2
  Cross-qu-quart gates: 0
Result: QuQuartCircuit(2 qu-quarts from 4 qubits, 2 layers, 0 cross-qu-quart gates)
Note: with no 2-qubit gates, any mapping is equivalent

Case B: Circuit with isolated qubit (q3 never used in 2-qubit gate)

No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {(0, 1): 1, (1, 2): 1}

  Most optimal pairing: [(0, 1), (2, 3)]
  Cross-qu-quart gates: 1

Converting 3 layers...
  Layer 1: Cross-qu-quart gate on qubits [1, 2] -> qu-quarts [0, 1]

Conversion complete!
  Total layers: 3
  Cross-qu-quart gates: 1
Result: QuQuartCircuit(2 qu-quarts from 4 qubits, 3 layers, 1 cross-qu-quart gates)
Note: q3 will be paired with anot

## Example 6: 6-Qubit Circuit → 3 QuQuarts
BMO should find `[(0,1), (2,3), (4,5)]` due to strong intra-pair coupling.

In [7]:
H    = KNOWN_GATES["H"]
CNOT = KNOWN_GATES["CNOT"]
CZ   = KNOWN_GATES["CZ"]

circuit = QubitCircuit(num_qubits=6)

circuit.add_layer([(H, [i]) for i in range(6)])

# Repeat intra-pair coupling twice to reinforce the pairing signal for BMO
for _ in range(2):
    circuit.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3]), (CNOT, [4, 5])])

# One cross-pair gate
circuit.add_layer([(CZ, [1, 4])])

ququart_circuit = qubit_to_ququart_circuit(circuit)
display_ququart_circuit(ququart_circuit)

print(f"\nResult: {ququart_circuit}")
print(f"Expected mapping: [(0,1), (2,3), (4,5)]")


No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {(0, 1): 2, (1, 4): 1, (2, 3): 2, (4, 5): 2}

  Most optimal pairing: [(0, 1), (2, 3), (4, 5)]
  Cross-qu-quart gates: 1

Converting 4 layers...
  Layer 3: Cross-qu-quart gate on qubits [1, 4] -> qu-quarts [0, 2]

Conversion complete!
  Total layers: 4
  Cross-qu-quart gates: 1

QuQuart Circuit Details

Qubit-to-QuQuart Mapping (Optimal):

  QQ0: qubit 0 (pos 0) + qubit 1 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ0 = |0>_q0 |0>_q1  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ0 = |1>_q0 |0>_q1  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ0 = |0>_q0 |1>_q1  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ0 = |1>_q0 |1>_q1  (pos0=1, pos1=1 -> 1+2=3)

  QQ1: qubit 2 (pos 0) + qubit 3 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ1 = |0>_q2 |0>_q3  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ1 = |1>_q2 |0>_q3  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ1 = |0>_q2 |1>_q3  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ1 

## Example 7: 8-Qubit GHZ-Style Circuit → 4 QuQuarts
Local pairing `(0,1), (2,3), (4,5), (6,7)` with cross-ququart entanglement at the end.

In [8]:
H    = KNOWN_GATES["H"]
CNOT = KNOWN_GATES["CNOT"]

circuit = QubitCircuit(num_qubits=8)

circuit.add_layer([(H, [0])])

for _ in range(3):
    circuit.add_layer([
        (CNOT, [0, 1]),
        (CNOT, [2, 3]),
        (CNOT, [4, 5]),
        (CNOT, [6, 7]),
    ])

circuit.add_layer([
    (CNOT, [1, 2]),   # connects ququart 0 and ququart 1
    (CNOT, [5, 6]),   # connects ququart 2 and ququart 3
])

ququart_circuit = qubit_to_ququart_circuit(circuit)
display_ququart_circuit(ququart_circuit)

print(f"\nResult: {ququart_circuit}")
print(f"Expected mapping: [(0,1), (2,3), (4,5), (6,7)]")


No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {(0, 1): 3, (1, 2): 1, (2, 3): 3, (4, 5): 3, (5, 6): 1, (6, 7): 3}

  Most optimal pairing: [(0, 1), (2, 3), (4, 5), (6, 7)]
  Cross-qu-quart gates: 2

Converting 5 layers...
  Layer 4: Cross-qu-quart gate on qubits [1, 2] -> qu-quarts [0, 1]
  Layer 4: Cross-qu-quart gate on qubits [5, 6] -> qu-quarts [2, 3]

Conversion complete!
  Total layers: 5
  Cross-qu-quart gates: 2

QuQuart Circuit Details

Qubit-to-QuQuart Mapping (Optimal):

  QQ0: qubit 0 (pos 0) + qubit 1 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ0 = |0>_q0 |0>_q1  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ0 = |1>_q0 |0>_q1  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ0 = |0>_q0 |1>_q1  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ0 = |1>_q0 |1>_q1  (pos0=1, pos1=1 -> 1+2=3)

  QQ1: qubit 2 (pos 0) + qubit 3 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ1 = |0>_q2 |0>_q3  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ1 = |1>_q2 |0>

## Example 8: 6-Qubit Circuit with Manual 3-QuQuart Mapping
Compares a hand-chosen mapping against the BMO-optimized mapping.

In [9]:
CNOT = KNOWN_GATES["CNOT"]
CZ   = KNOWN_GATES["CZ"]

circuit = QubitCircuit(num_qubits=6)
circuit.add_layer([(CNOT, [0, 2]), (CNOT, [3, 5])])
circuit.add_layer([(CZ,   [1, 3]), (CZ,   [2, 4])])
circuit.add_layer([(CNOT, [0, 4])])
circuit.add_layer([(CNOT, [1, 5])])

manual_mapping = [(0, 2), (1, 3), (4, 5)]

print(f"Manual mapping: {manual_mapping}")
ququart_manual = qubit_to_ququart_circuit(circuit, mapping=manual_mapping)
display_ququart_circuit(ququart_manual)
print(f"\nManual result: {ququart_manual}")

print("\nNow running BMO for comparison...")
ququart_bmo = qubit_to_ququart_circuit(circuit)
print(f"BMO result:    {ququart_bmo}")
print(f"\nManual mapping cross-gates: {ququart_manual.cross_ququart_gate_count}")
print(f"BMO mapping cross-gates:    {ququart_bmo.cross_ququart_gate_count}")

Manual mapping: [(0, 2), (1, 3), (4, 5)]
Using provided mapping: [(0, 2), (1, 3), (4, 5)]

Converting 4 layers...
  Layer 0: Cross-qu-quart gate on qubits [3, 5] -> qu-quarts [1, 2]
  Layer 1: Cross-qu-quart gate on qubits [2, 4] -> qu-quarts [0, 2]
  Layer 2: Cross-qu-quart gate on qubits [0, 4] -> qu-quarts [0, 2]
  Layer 3: Cross-qu-quart gate on qubits [1, 5] -> qu-quarts [1, 2]

Conversion complete!
  Total layers: 4
  Cross-qu-quart gates: 4

QuQuart Circuit Details

Qubit-to-QuQuart Mapping (Optimal):

  QQ0: qubit 0 (pos 0) + qubit 2 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ0 = |0>_q0 |0>_q2  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ0 = |1>_q0 |0>_q2  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ0 = |0>_q0 |1>_q2  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ0 = |1>_q0 |1>_q2  (pos0=1, pos1=1 -> 1+2=3)

  QQ1: qubit 1 (pos 0) + qubit 3 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ1 = |0>_q1 |0>_q3  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ1 = |1>_q1 |0>_q3  (pos0=1, pos1=0 -> 1+0=1)
    

## Example 9: Odd Qubit Counts — Ancilla Added Automatically
For an odd number of qubits, BMO injects one ancilla qubit so every qubit forms a complete ququart pair.

In [10]:
CNOT = KNOWN_GATES["CNOT"]
H    = KNOWN_GATES["H"]

# --- 3-qubit GHZ circuit → 2 ququarts (1 ancilla) ---
print("--- 3-qubit GHZ circuit → 2 ququarts ---")
circuit_3 = QubitCircuit(num_qubits=3)
circuit_3.add_layer([(H, [0])])
circuit_3.add_layer([(CNOT, [0, 1])])
circuit_3.add_layer([(CNOT, [1, 2])])

qqc_3 = qubit_to_ququart_circuit(circuit_3)
display_ququart_circuit(qqc_3)
print(f"Result: {qqc_3}")
print("Note: one ququart pair includes the ancilla qubit")

# --- 5-qubit circuit → 3 ququarts (1 ancilla) ---
print("\n--- 5-qubit circuit → 3 ququarts ---")
circuit_5 = QubitCircuit(num_qubits=5)
circuit_5.add_layer([(H, [i]) for i in range(5)])
circuit_5.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3])])
circuit_5.add_layer([(CNOT, [0, 1]), (CNOT, [2, 3])])
circuit_5.add_layer([(CNOT, [1, 4])])   # cross-pair gate

qqc_5 = qubit_to_ququart_circuit(circuit_5)
display_ququart_circuit(qqc_5)
print(f"Result: {qqc_5}")

--- 3-qubit GHZ circuit → 2 ququarts ---

No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis (odd qubit count — ancilla qubit 3 added):
  Interaction counts: {(0, 1): 1, (1, 2): 1}

  Most optimal pairing: [(0, 1), (2, 3)]
  Cross-qu-quart gates: 1

Converting 3 layers...
  Layer 2: Cross-qu-quart gate on qubits [1, 2] -> qu-quarts [0, 1]

Conversion complete!
  Total layers: 3
  Cross-qu-quart gates: 1

QuQuart Circuit Details

Qubit-to-QuQuart Mapping (Optimal):

  QQ0: qubit 0 (pos 0) + qubit 1 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ0 = |0>_q0 |0>_q1  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ0 = |1>_q0 |0>_q1  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ0 = |0>_q0 |1>_q1  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ0 = |1>_q0 |1>_q1  (pos0=1, pos1=1 -> 1+2=3)

  QQ1: qubit 2 (pos 0) + qubit 3 (pos 1) [ancilla]
    level = pos0_val + 2*pos1_val
    |0>_QQ1 = |0>_q2 |0>_q3  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ1 = |1>_q2 |0>_q3  (pos0=1, pos1=0 -> 1+0=1)
    

## Example 10: 7-Qubit Circuit → 4 QuQuarts
Odd count with 7 qubits — BMO adds one ancilla to form 4 complete ququart pairs.

In [11]:
circuit7 = QubitCircuit(num_qubits=8)

circuit7.add_layer([(KNOWN_GATES["H"], [0])]) #First layer in this circuit

#Each tuple () = one gate being executed on 1 or 2 qubits
#In each tuple (KNOWN_GATES["CNOT"], [0, 1]) = (CNOT GATE, ACTING on qubits 0, 1)
circuit7.add_layer([(KNOWN_GATES["CNOT"], [0, 2]), (KNOWN_GATES["CNOT"], [1, 3]), (KNOWN_GATES["CNOT"], [4, 5])]) 
circuit7.add_layer([(KNOWN_GATES["CZ"], [0, 6]), (KNOWN_GATES["CY"], [2, 3])])

qqc7 = qubit_to_ququart_circuit(circuit7)
display_ququart_circuit(qqc7)

print(f"\nResult: {qqc7}")


No mapping was provided. We'll now run the Best Mapping Optimization...

BMO Analysis:
  Interaction counts: {(0, 2): 1, (0, 6): 1, (1, 3): 1, (2, 3): 1, (4, 5): 1}

  Most optimal pairing: [(0, 2), (1, 3), (4, 5), (6, 7)]
  Cross-qu-quart gates: 2

Converting 3 layers...
  Layer 2: Cross-qu-quart gate on qubits [0, 6] -> qu-quarts [0, 3]
  Layer 2: Cross-qu-quart gate on qubits [2, 3] -> qu-quarts [0, 1]

Conversion complete!
  Total layers: 3
  Cross-qu-quart gates: 2

QuQuart Circuit Details

Qubit-to-QuQuart Mapping (Optimal):

  QQ0: qubit 0 (pos 0) + qubit 2 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ0 = |0>_q0 |0>_q2  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ0 = |1>_q0 |0>_q2  (pos0=1, pos1=0 -> 1+0=1)
    |2>_QQ0 = |0>_q0 |1>_q2  (pos0=0, pos1=1 -> 0+2=2)
    |3>_QQ0 = |1>_q0 |1>_q2  (pos0=1, pos1=1 -> 1+2=3)

  QQ1: qubit 1 (pos 0) + qubit 3 (pos 1)
    level = pos0_val + 2*pos1_val
    |0>_QQ1 = |0>_q1 |0>_q3  (pos0=0, pos1=0 -> 0+0=0)
    |1>_QQ1 = |1>_q1 |0>_q3  (pos0=